# Running LLMs on a Laptop: GGUF and llama.cpp

The runnable code of [Running LLMs on a Laptop: GGUF and llama.cpp](https://residual-stream.lvtan71.workers.dev/posts/running-llms-on-a-laptop/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/running-llms-on-a-laptop.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 1 min end to end.
The post's speed numbers are from llama.cpp on Apple's Metal GPU; on Colab, llama-cpp-python runs on the CPU (it takes several minutes to install), so expect the quality numbers to match and the speeds not to.

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
%pip install -q llama-cpp-python gguf

# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "running-llms-on-a-laptop"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in ['shared/running-llms-on-a-laptop/corpus.py', 'data/blog-ed51ca8.tar.gz']:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## What is in a GGUF file

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/running-llms-on-a-laptop/#what-is-in-a-gguf-file)

In [ ]:
import collections
import os
from gguf import GGUFReader
from huggingface_hub import snapshot_download

# What is inside a GGUF file: a header, key-value metadata (architecture,
# hyperparameters, the tokenizer itself), then a table of tensors, each
# with a name, shape, type and offset, then the tensor data, aligned so it
# can be memory-mapped straight from disk.
D = snapshot_download("Qwen/Qwen2.5-1.5B-Instruct-GGUF",
                      revision="91cad51170dc346986eccefdc2dd33a9da36ead9",
                      allow_patterns=["*q3_k_m.gguf", "*q4_k_m.gguf", "*q8_0.gguf"]) + "/"
r = GGUFReader(D + "qwen2.5-1.5b-instruct-q4_k_m.gguf")
fields = r.fields
def value(key):
    f = fields[key]
    return f.parts[f.data[0]].tobytes().decode() if f.types and f.types[0].name == "STRING" else f.parts[f.data[0]][0]
kv = [k for k in fields if not k.startswith("GGUF.")]      # the reader adds GGUF.version and the counts itself
print(f"qwen2.5-1.5b-instruct-q4_k_m.gguf: {len(kv)} metadata keys, {len(r.tensors)} tensors")
for key in ("general.architecture", "qwen2.block_count", "qwen2.context_length", "qwen2.embedding_length",
            "qwen2.attention.head_count", "qwen2.attention.head_count_kv", "general.file_type"):
    print(f"  {key:32s} {value(key)}")
print(f"  {'tokenizer.ggml.tokens':32s} {len(fields['tokenizer.ggml.tokens'].data):,} entries (the vocabulary, in the file)")
by_type = collections.defaultdict(lambda: [0, 0, 0])
for t in r.tensors:
    b = by_type[t.tensor_type.name]
    b[0] += 1; b[1] += int(t.n_elements); b[2] += int(t.n_bytes)
print("tensors by type:")
for name, (n, elems, nbytes) in sorted(by_type.items(), key=lambda kv: -kv[1][2]):
    print(f"  {name:6s} {n:4d} tensors {elems / 1e6:8.1f}M weights {nbytes / 2**20:8.1f} MiB  {8 * nbytes / elems:5.2f} bits per weight")
print("examples:", ", ".join(f"{t.name} {t.tensor_type.name}" for t in r.tensors[:6]))
q6 = [t.name for t in r.tensors if t.tensor_type.name == "Q6_K"]
print(f"the {len(q6)} Q6_K tensors: output.weight, {sum('attn_v' in n for n in q6)} attn_v and "
      f"{sum('ffn_down' in n for n in q6)} ffn_down (layers {sorted({int(n.split('.')[1]) for n in q6 if n.startswith('blk')})})")
for f in sorted(os.listdir(D)):
    if f.endswith(".gguf"):
        rr = GGUFReader(D + f)
        n = sum(int(t.n_elements) for t in rr.tensors)
        print(f"{f:36s} {os.path.getsize(D + f) / 1e9:5.2f} GB, {8 * sum(int(t.n_bytes) for t in rr.tensors) / n:5.2f} bits per weight on average")

The post printed:

```text
qwen2.5-1.5b-instruct-q4_k_m.gguf: 26 metadata keys, 339 tensors
  general.architecture             qwen2
  qwen2.block_count                28
  qwen2.context_length             32768
  qwen2.embedding_length           1536
  qwen2.attention.head_count       12
  qwen2.attention.head_count_kv    2
  general.file_type                15
  tokenizer.ggml.tokens            151,936 entries (the vocabulary, in the file)
tensors by type:
  Q4_K    169 tensors   1345.4M weights    721.7 MiB   4.50 bits per weight
  Q6_K     29 tensors    431.6M weights    337.6 MiB   6.56 bits per weight
  F32     141 tensors      0.1M weights      0.6 MiB  32.00 bits per weight
examples: output.weight Q6_K, token_embd.weight Q4_K, blk.0.attn_norm.weight F32, blk.0.ffn_down.weight Q6_K, blk.0.ffn_gate.weight Q4_K, blk.0.ffn_up.weight Q4_K
the 29 Q6_K tensors: output.weight, 14 attn_v and 14 ffn_down (layers [0, 1, 5, 6, 7, 8, 9, 10, 13, 16, 19, 21, 24, 27])
qwen2.5-1.5b-instruct-q3_k_m.gguf     0.92 GB,  4.13 bits per weight on average
qwen2.5-1.5b-instruct-q4_k_m.gguf     1.12 GB,  5.00 bits per weight on average
qwen2.5-1.5b-instruct-q8_0.gguf       1.89 GB,  8.50 bits per weight on average
```

## Speed: bandwidth divided by bytes

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/running-llms-on-a-laptop/#speed-bandwidth-divided-by-bytes)

In [ ]:
import time
from gguf import GGUFReader
from llama_cpp import Llama
from huggingface_hub import snapshot_download

# Tokens per second, measured against the bandwidth bound of "Small and Specialized Models":
# each decoded token reads every weight once, so tokens/s <= bandwidth /
# bytes of weights. The bytes read per token are every tensor except the
# token-embedding table, of which a token reads one row. This laptop's GPU
# measured 93 GB/s in "How a GPU Runs Deep Learning". Decode
# speed is timed from the first generated token to the last (128 tokens,
# greedy), so the prompt's prefill is excluded; median of 3 runs. The model
# is locked in RAM (use_mlock), so the OS can't page it out mid-run.
D = snapshot_download("Qwen/Qwen2.5-1.5B-Instruct-GGUF",
                      revision="91cad51170dc346986eccefdc2dd33a9da36ead9",
                      allow_patterns=["*q3_k_m.gguf", "*q4_k_m.gguf", "*q8_0.gguf"]) + "/"
BW = 93e9
prompt = "<|im_start|>user\nExplain what a KV cache is, in detail.<|im_end|>\n<|im_start|>assistant\n"
def bytes_per_token(path):
    tensors = GGUFReader(path).tensors
    emb = next(t for t in tensors if t.name == "token_embd.weight")
    return sum(int(t.n_bytes) for t in tensors) - int(emb.n_bytes) * (1 - 1 / emb.shape[-1])

print(f"{'file':36s} {'GB read/token':>13s} {'bound':>9s} {'GPU (Metal)':>12s} {'of bound':>9s} {'CPU, 8 threads':>15s}")
for f in ("qwen2.5-1.5b-instruct-q8_0.gguf", "qwen2.5-1.5b-instruct-q4_k_m.gguf", "qwen2.5-1.5b-instruct-q3_k_m.gguf"):
    size = bytes_per_token(D + f)
    speeds = {}
    for name, layers in (("gpu", -1), ("cpu", 0)):
        llm = Llama(D + f, n_gpu_layers=layers, n_ctx=1024, n_threads=8, verbose=False, seed=0, use_mlock=True)
        runs = []
        for _ in range(3):
            stamps = []
            for _ in llm(prompt, max_tokens=128, temperature=0.0, top_k=1, repeat_penalty=1.0, stream=True):
                stamps.append(time.perf_counter())
            runs.append((len(stamps) - 1) / (stamps[-1] - stamps[0]))
        speeds[name] = sorted(runs)[1]
        del llm
    print(f"{f:36s} {size / 1e9:11.2f}   {BW / size:6.0f}/s {speeds['gpu']:9.0f}/s {speeds['gpu'] / (BW / size):8.0%} "
          f"{speeds['cpu']:12.0f}/s")

The post printed:

```text
file                                 GB read/token     bound  GPU (Metal)  of bound  CPU, 8 threads
qwen2.5-1.5b-instruct-q8_0.gguf             1.64       57/s        57/s     101%           62/s
qwen2.5-1.5b-instruct-q4_k_m.gguf           0.98       95/s        86/s      90%           90/s
qwen2.5-1.5b-instruct-q3_k_m.gguf           0.82      114/s        73/s      64%           85/s
```

## Quality: what each level costs

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/running-llms-on-a-laptop/#quality-what-each-level-costs)

In [ ]:
import math
import sys
import numpy as np
from gguf import GGUFReader
from llama_cpp import Llama
from huggingface_hub import snapshot_download

# What each quantization costs in quality: bits per token on two of this
# blog's posts, written after the model was trained (one 1,024-token window
# from the start of each, the second half of each window scored), computed
# from llama.cpp's own logits for each file.
D = snapshot_download("Qwen/Qwen2.5-1.5B-Instruct-GGUF",
                      revision="91cad51170dc346986eccefdc2dd33a9da36ead9",
                      allow_patterns=["*q3_k_m.gguf", "*q4_k_m.gguf", "*q8_0.gguf"]) + "/"
sys.path.insert(0, "../post7")
from corpus import clean
POSTS = "content/posts/"
texts = [clean(open(POSTS + f + ".mdx").read()) for f in ("kv-cache-explained", "bpe-tokenizer-explained")]
emb = next(t for t in GGUFReader(D + "qwen2.5-1.5b-instruct-q8_0.gguf").tensors if t.name == "token_embd.weight")
print("token_embd shape in the file:", list(emb.shape))
for f in ("qwen2.5-1.5b-instruct-q8_0.gguf", "qwen2.5-1.5b-instruct-q4_k_m.gguf", "qwen2.5-1.5b-instruct-q3_k_m.gguf"):
    llm = Llama(D + f, n_gpu_layers=-1, n_ctx=1024, logits_all=True, verbose=False)
    total, n = 0.0, 0
    for text in texts:
        seq = llm.tokenize(text.encode(), add_bos=False)[:1024]
        llm.reset(); llm.eval(seq)
        logits = np.array(llm.scores[:len(seq)])
        logp = logits - np.log(np.exp(logits - logits.max(-1, keepdims=True)).sum(-1, keepdims=True)) - logits.max(-1, keepdims=True)
        for i in range(512, len(seq)):
            total -= logp[i - 1, seq[i]]; n += 1
    print(f"{f:36s} {total / n / math.log(2):.3f} bits per token")
    del llm

The post printed:

```text
token_embd shape in the file: [np.uint64(1536), np.uint64(151936)]
qwen2.5-1.5b-instruct-q8_0.gguf      3.061 bits per token
qwen2.5-1.5b-instruct-q4_k_m.gguf    3.114 bits per token
qwen2.5-1.5b-instruct-q3_k_m.gguf    3.185 bits per token
```